# Employee Salary Prediction - Model Training

## Objective

This notebook builds, trains, and evaluates multiple machine learning regression
models for predicting employee salaries, and documents the reasoning behind
**which model is deployed to the live website** versus which model is kept as
the best offline / research model.

### Models Used

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

### Evaluation Metrics

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score

### Deployment Note (read this first)

Two models are trained and compared below. **Linear Regression is the model
actually deployed on the live website**, while **Random Forest is kept as the
best-performing offline model** for anyone who wants maximum accuracy and is
able to train/host a larger model file. The reasoning and full metric
comparison are documented in **Section 12** and **Section 16**.


## 1. Import Libraries

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np
import joblib
import os

## 2. Load the Cleaned Dataset

In [3]:
data = pd.read_csv("../data/employees_dataset_cleaned.csv")
df = pd.DataFrame(data)
df

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069
...,...,...,...,...,...,...,...,...,...,...
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467


## 3. Separate Features and Target Variable

The target variable is **salary**; all remaining columns are used as input features.


In [4]:
x = df.drop("salary", axis=1)
y = df["salary"]

## 4. Identify Numerical and Categorical Features

In [5]:
num_features = ["experience_years", "skills_count", "certifications"]
cat_features = ["job_title", "education_level", "industry", "company_size", "location", "remote_work"]
print(x.dtypes)

job_title           object
experience_years     int64
education_level     object
skills_count         int64
industry            object
company_size        object
location            object
remote_work         object
certifications       int64
dtype: object


## 5. Split Dataset into Training and Testing Sets

An 80:20 train/test split is used, with a fixed `random_state` for reproducibility.


In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(200000, 9) (50000, 9) (200000,) (50000,)


## 6. Data Preprocessing Pipeline

### Numerical Pipeline
Numerical features are:
- Imputed using the median strategy
- Standardized using `StandardScaler`


In [7]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### Categorical Pipeline
Categorical features are encoded using `OneHotEncoder`.


In [8]:
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### Combine Pipelines using ColumnTransformer

In [9]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

## 7. Transform Training and Testing Data

In [10]:
x_train_prepared = preprocessor.fit_transform(x_train)
x_test_prepared = preprocessor.transform(x_test)
print("Train shape:", x_train_prepared.shape)
print("Test shape :", x_test_prepared.shape)

Train shape: (200000, 48)
Test shape : (50000, 48)


## 8. Train Model — Linear Regression

Linear Regression is trained as the lightweight baseline model. As shown in
Section 12, this is the model that ends up being deployed on the live website
because of its tiny file size and near-instant load time, so it is trained and
evaluated carefully here rather than being treated as a throwaway baseline.


In [11]:
lin_reg = LinearRegression()
lin_reg.fit(x_train_prepared, y_train)
lin_reg_predictions = lin_reg.predict(x_test_prepared)

### Evaluation — Linear Regression

In [12]:
lin_mae = mean_absolute_error(y_test, lin_reg_predictions)
lin_mse = mean_squared_error(y_test, lin_reg_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_r2 = r2_score(y_test, lin_reg_predictions)

print("Linear Regression Performance")
print("MAE  :", lin_mae)
print("RMSE :", lin_rmse)
print("R²   :", lin_r2)

Linear Regression Performance
MAE  : 5436.096958649476
RMSE : 7125.522920575376
R²   : 0.9634690226760201


## 9. Train Model — Decision Tree Regressor

In [13]:
dec_reg = DecisionTreeRegressor(random_state=42)
dec_reg.fit(x_train_prepared, y_train)
dec_reg_predictions = dec_reg.predict(x_test_prepared)

### Evaluation — Decision Tree Regressor

In [14]:
dec_mae = mean_absolute_error(y_test, dec_reg_predictions)
dec_mse = mean_squared_error(y_test, dec_reg_predictions)
dec_rmse = np.sqrt(dec_mse)
dec_r2 = r2_score(y_test, dec_reg_predictions)

print("Decision Tree Performance")
print("MAE  :", dec_mae)
print("RMSE :", dec_rmse)
print("R²   :", dec_r2)

Decision Tree Performance
MAE  : 7286.74165
RMSE : 9228.783604854705
R²   : 0.9387202853928697


## 10. Train Model — Random Forest Regressor

Random Forest combines multiple decision trees to improve accuracy and reduce
overfitting. This is the **best-performing model** in this project, and is
kept available for anyone running the project locally / offline who wants
maximum prediction accuracy and can afford a larger model file and longer
load time (see Section 12 for why it is *not* used on the live website).


In [15]:
random_forest_reg = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42)
random_forest_reg.fit(x_train_prepared, y_train)
rf_predictions = random_forest_reg.predict(x_test_prepared)

### Evaluation — Random Forest Regressor

In [16]:
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Performance")
print("MAE  :", rf_mae)
print("RMSE :", rf_rmse)
print("R²   :", rf_r2)

Random Forest Performance
MAE  : 5048.546023452857
RMSE : 6364.544987946304
R²   : 0.9708551026755426


## 11. Model Comparison

Performance of all three trained models, evaluated on the same held-out test set.


In [17]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE": [lin_mae, dec_mae, rf_mae],
    "RMSE": [lin_rmse, dec_rmse, rf_rmse],
    "R2 Score": [lin_r2, dec_r2, rf_r2]
})
results

,Model,MAE,RMSE,R2 Score
0,Linear Regression,5436.096959,7125.522921,0.963469
1,Decision Tree,7286.741650,9228.783605,0.938720
2,Random Forest,5048.546023,6364.544988,0.970855


## 12. Model Selection: Best Model vs. Deployed Model

Two different questions are being answered here, and they have two different answers:

**Q1: Which model performs best overall?**
Random Forest — it has the lowest MAE, lowest RMSE, and highest R² of the three models.

**Q2: Which model is deployed on the live website?**
Linear Regression — for practical deployment reasons explained below.

### Why Random Forest is the best model (offline)

| Metric | Linear Regression | Decision Tree | Random Forest |
|---|---:|---:|---:|
| MAE | ~5,436 | ~7,292 | **~5,049 (best)** |
| RMSE | ~7,126 | ~9,251 | **~6,365 (best)** |
| R² | ~0.9635 | ~0.9384 | **~0.9709 (best)** |

Random Forest wins on every metric, so if the goal is purely the most accurate
predictions, it is the correct choice.

### Why Linear Regression is deployed on the live website instead

The trained Random Forest model file is **~887 MB**. On a live/hosted web
app this causes real, practical problems:

- **Slow or failed cold starts** — most free/low-tier hosting platforms
  (Streamlit Community Cloud, Render free tier, Hugging Face Spaces free
  tier, etc.) have RAM and disk limits well under 1 GB, so an 887 MB model
  can fail to load at all.
- **Slow first response** — even where it does load, unpickling an
  887 MB file adds many seconds (sometimes minutes) to the first request.
- **Repo/storage limits** — many git-based deployment platforms reject or
  strain under files that large, and GitHub itself blocks files over 100 MB
  without Git LFS.
- **Cost** — larger memory footprint often means a paid hosting tier is
  required just to keep the model in memory.

The Linear Regression model, by contrast, is only a few KB, loads instantly,
and — critically — **its accuracy is very close to Random Forest's**:

- R² difference: **0.9709 vs 0.9635** → about **0.74 percentage points**
- MAE difference: **~5,049 vs ~5,436** → about **₹387** on an average
  salary of ~₹1,45,000 (roughly a 0.27% difference in absolute terms)

In other words, Random Forest is only marginally more accurate, but Linear
Regression is dramatically cheaper to deploy. For a live, publicly-hosted
salary predictor, that tradeoff clearly favors Linear Regression.

### Decision

- ✅ **Linear Regression** → deployed on the live website (`preprocessor.pkl`
  + `linear_regression_salary_model.pkl`)
- ✅ **Random Forest** → saved separately as the best-performing model,
  for local/offline use by anyone who wants the highest possible accuracy
  and is able to host or run a larger model file

> **If you want to use Random Forest instead of Linear Regression:**
> you do not need to retrain it — the trained Random Forest model is saved
> in this notebook (Section 16) as `random_forest_salary_model.pkl`. Simply
> load that file instead of the Linear Regression file in the web app's
> inference code, using the *same* `preprocessor.pkl` for preprocessing. If
> you'd rather train it fresh (e.g. after changing data or hyperparameters),
> Section 10 above contains the full training code.


## 13. Optional: Hyperparameter Tuning for Random Forest

Since Random Forest is kept as the "best accuracy" option for offline use,
it is still worth tuning. This step is optional and only relevant to anyone
using the Random Forest path.


In [18]:
random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions={
        "n_estimators": [50, 100, 150, 200],
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    },
    n_iter=10,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train_prepared, y_train)

print("Best Parameters:")
print(random_search.best_params_)
print("\nBest Cross Validation R² Score:")
print(random_search.best_score_)

Best Parameters:
{'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 30}

Best Cross Validation R² Score:
0.9676678268521144


In [19]:
best_rf_model = random_search.best_estimator_
tuned_rf_predictions = best_rf_model.predict(x_test_prepared)

tuned_rf_mae = mean_absolute_error(y_test, tuned_rf_predictions)
tuned_rf_mse = mean_squared_error(y_test, tuned_rf_predictions)
tuned_rf_rmse = np.sqrt(tuned_rf_mse)
tuned_rf_r2 = r2_score(y_test, tuned_rf_predictions)

print("Tuned Random Forest — Test Set Performance")
print("MAE  :", tuned_rf_mae)
print("RMSE :", tuned_rf_rmse)
print("R²   :", tuned_rf_r2)

Tuned Random Forest — Test Set Performance
MAE  : 5097.757087881144
RMSE : 6437.156278809632
R²   : 0.9701862974549542


Compare the default Random Forest against the tuned version, and keep
whichever generalizes better on the held-out test set.


In [20]:
if tuned_rf_r2 >= rf_r2:
    final_rf_model = best_rf_model
    final_rf_predictions = tuned_rf_predictions
    print("Selected: Tuned Random Forest (better test-set R²)")
else:
    final_rf_model = random_forest_reg
    final_rf_predictions = rf_predictions
    print("Selected: Default Random Forest (tuning did not improve test-set R²)")

Selected: Default Random Forest (tuning did not improve test-set R²)


## 14. Actual vs Predicted Salary — Deployed Model (Linear Regression)

This comparison reflects what website users will actually experience, since
Linear Regression is the model running live.


In [21]:
comparison_lin = pd.DataFrame({
    "Actual Salary": y_test.values,
    "Predicted Salary": lin_reg_predictions
})
comparison_lin["Difference"] = comparison_lin["Actual Salary"] - comparison_lin["Predicted Salary"]
comparison_lin["Percentage Error"] = (
    abs(comparison_lin["Difference"]) / comparison_lin["Actual Salary"]
) * 100

comparison_lin.head(15)

,Actual Salary,Predicted Salary,Difference,Percentage Error
0,164009,172850.846941,-8841.846941,5.391074
1,79594,89234.518542,-9640.518542,12.112117
2,74090,63791.551344,10298.448656,13.899917
3,177193,168311.829996,8881.170004,5.012145
4,120012,117222.924014,2789.075986,2.323998
5,163369,171755.343207,-8386.343207,5.133375
6,111889,110424.891810,1464.108190,1.308536
7,75418,60449.432924,14968.567076,19.847473
8,103067,93094.302983,9972.697017,9.675936
9,190692,192448.962239,-1756.962239,0.921361


In [26]:
os.makedirs("../data", exist_ok=True)
comparison_lin.to_csv("salary_predictions_linear_regression.csv", index=False)

## 15. Actual vs Predicted Salary — Best Offline Model (Random Forest)

Saved separately so both files are available: one showing what the live
website users see, and one showing the best achievable accuracy offline.


In [27]:
comparison_rf = pd.DataFrame({
    "Actual Salary": y_test.values,
    "Predicted Salary": final_rf_predictions
})
comparison_rf["Difference"] = comparison_rf["Actual Salary"] - comparison_rf["Predicted Salary"]
comparison_rf["Percentage Error"] = (
    abs(comparison_rf["Difference"]) / comparison_rf["Actual Salary"]
) * 100

comparison_rf.head(15)

,Actual Salary,Predicted Salary,Difference,Percentage Error
0,164009,167861.68,-3852.68,2.349066
1,79594,92881.16,-13287.16,16.693670
2,74090,70176.00,3914.00,5.282764
3,177193,162563.08,14629.92,8.256489
4,120012,115070.18,4941.82,4.117772
5,163369,163882.26,-513.26,0.314172
6,111889,110474.78,1414.22,1.263949
7,75418,73300.76,2117.24,2.807340
8,103067,96603.02,6463.98,6.271629
9,190692,196073.36,-5381.36,2.822017


In [28]:
comparison_rf.to_csv("../data/salary_predictions_random_forest.csv", index=False)

## 16. Saving the Models

Both models and the shared preprocessing pipeline are saved with **Joblib**.

Files saved to the `models/` directory:

- **`preprocessor.pkl`** — fitted preprocessing pipeline (imputer, scaler,
  encoder). Required by *both* models below — always load this alongside
  whichever model you use.
- **`linear_regression_salary_model.pkl`** — small, fast-loading model.
  **This is the model used by the live website.**
- **`random_forest_salary_model.pkl`** — large (~887 MB), highest-accuracy
  model. Kept for local/offline use by anyone who wants maximum accuracy.
  Not used in the live deployment due to its size (see Section 12).

### How the web app should load the model

```python
import joblib

preprocessor = joblib.load("models/preprocessor.pkl")

# Deployed on the live website:
model = joblib.load("models/linear_regression_salary_model.pkl")

# To use the more accurate (but much larger) model instead, simply swap the
# line above for:
# model = joblib.load("models/random_forest_salary_model.pkl")
```


In [29]:
os.makedirs("../models", exist_ok=True)

# Shared preprocessing pipeline (used by both models)
joblib.dump(preprocessor, "../models/preprocessor.pkl")

# Deployed model: Linear Regression
joblib.dump(lin_reg, "../models/linear_regression_salary_model.pkl")

# Best-performing offline model: Random Forest
joblib.dump(final_rf_model, "../models/random_forest_salary_model.pkl")

print("Preprocessor, Linear Regression model, and Random Forest model saved successfully.")

Preprocessor, Linear Regression model, and Random Forest model saved successfully.


## 17. Summary

| | Deployed on live website | Best offline model |
|---|---|---|
| **Model** | Linear Regression | Random Forest |
| **R²** | ~0.9635 | ~0.9709 |
| **MAE** | ~5,436 | ~5,049 |
| **File size** | a few KB | ~887 MB |
| **Load time** | instant | several seconds–minutes, depending on host |
| **File** | `linear_regression_salary_model.pkl` | `random_forest_salary_model.pkl` |

**Bottom line:** Linear Regression is used in production because it delivers
almost the same accuracy as Random Forest (within ~0.74 R² points / ~₹387
MAE) at a tiny fraction of the size and load time — which matters far more
than a marginal accuracy gain when the model needs to load reliably on a
live, publicly hosted website. Random Forest remains available in the
`models/` folder for anyone who prefers maximum accuracy over deployment
convenience.
